# Model 2: Qwen2.5-3B + RAG (LangChain + FAISS + TriviaQA KB)
## RAG-Augmented QA with ROUGE & Hallucination Evaluation

### Design
- **Fine-tuned on**: SQuAD v2 (baseline.ipynb)
- **RAG Knowledge Base**: TriviaQA evidence passages (separate from fine-tune data!)
- **Evaluated on**: TriviaQA validation split

### Why this separation matters
The model was fine-tuned on SQuAD v2, so it has NOT memorized TriviaQA answers.
When RAG retrieves TriviaQA passages, the model must genuinely use that context.
This makes the faithfulness and hallucination comparison meaningful and fair.

```
TriviaQA Val Question
        ↓
   FAISS Retriever (TriviaQA KB)
        ↓ top-3 passages
   Fine-tuned Qwen2.5-3B
        ↓
   Answer → ROUGE + Faithfulness vs retrieved context
```

In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"]   = "1"
print("✅ CUDA debug flags set")

✅ CUDA debug flags set


In [2]:
import torch
import transformers, bitsandbytes, peft, trl

print("📦 Package versions:")
print(f"   torch          : {torch.__version__}")
print(f"   transformers   : {transformers.__version__}")
print(f"   bitsandbytes   : {bitsandbytes.__version__}")
print(f"   peft           : {peft.__version__}")
print(f"   trl            : {trl.__version__}")
print(f"   CUDA           : {torch.version.cuda}")
print()
if torch.cuda.is_available():
    print(f"   Device : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("✅ All checks passed!")

📦 Package versions:
   torch          : 2.7.0+cu128
   transformers   : 4.50.3
   bitsandbytes   : 0.49.2
   peft           : 0.14.0
   trl            : 0.13.0
   CUDA           : 12.8

   Device : NVIDIA GeForce RTX 5060 Laptop GPU
   VRAM   : 8.5 GB
✅ All checks passed!


In [3]:
import os, gc, json, torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document

from rouge_score import rouge_scorer
from sentence_transformers import CrossEncoder

# ── CONFIG ─────────────────────────────────────────────────────────────────────
BASE_MODEL          = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_DIR         = "/content/qwen25-squadv2"
MERGED_DIR          = "/content/qwen25-squadv2-merged"
FAISS_PATH          = "./faiss_triviaqa_bge"        # new path — forces rebuild with new embeddings
MODEL1_METRICS_PATH = "./model1_outputs/model1_metrics.json"
EMBED_MODEL         = "BAAI/bge-base-en-v1.5"       # upgraded from all-MiniLM-L6-v2
RERANKER_MODEL      = "cross-encoder/ms-marco-MiniLM-L-6-v2"
SAVE_DIR            = "./model2_outputs"

EVAL_SAMPLES  = 500
KB_SAMPLES    = 100000   # TriviaQA passages to index
TOP_K         = 5
TOP_K_RERANK  = 10 
MAX_SEQ_LEN   = 512
SEED          = 42

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)

print(f"🖥️  Device : {device} — {torch.cuda.get_device_name(0)}")
print(f"   VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB total")
print(f"   Fine-tune base : SQuAD v2 (from baseline.ipynb)")
print(f"   RAG KB         : TriviaQA evidence passages ({KB_SAMPLES:,} docs)")
print(f"   Eval set       : TriviaQA validation ({EVAL_SAMPLES} samples)")
print(f"   Embed model    : {EMBED_MODEL}")
print(f"   Reranker       : {RERANKER_MODEL}")
print(f"   Top-K (fetch)  : {TOP_K_RERANK}  →  Top-K (reranked): {TOP_K}")
print("\n✅ Config ready!")

🖥️  Device : cuda — NVIDIA GeForce RTX 5060 Laptop GPU
   VRAM   : 8.5 GB total
   Fine-tune base : SQuAD v2 (from baseline.ipynb)
   RAG KB         : TriviaQA evidence passages (100,000 docs)
   Eval set       : TriviaQA validation (500 samples)
   Embed model    : BAAI/bge-base-en-v1.5
   Reranker       : cross-encoder/ms-marco-MiniLM-L-6-v2
   Top-K (fetch)  : 10  →  Top-K (reranked): 5

✅ Config ready!


## Load TriviaQA — Build RAG Knowledge Base

In [4]:
print("📥 Loading TriviaQA (rc split)...")
triviaqa = load_dataset("trivia_qa", "rc")

for split, d in triviaqa.items():
    print(f"   {split:12s}: {len(d):,} samples")

ex = triviaqa['train'][0]
print(f"\n🔍 Sample question : {ex['question']}")
print(f"   Answer          : {ex['answer']['value']}")
print(f"   Evidence passages: {len(ex['search_results']['search_context'])} docs")
print(f"   First passage (200 chars): {ex['search_results']['search_context'][0][:200]}...")

📥 Loading TriviaQA (rc split)...


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/24 [00:00<?, ?it/s]

   train       : 138,384 samples
   validation  : 17,944 samples
   test        : 17,210 samples

🔍 Sample question : Which American-born Sinclair won the Nobel Prize for Literature in 1930?
   Answer          : Sinclair Lewis
   Evidence passages: 7 docs
   First passage (200 chars): The Nobel Prize in Literature 1930
The Nobel Prize in Literature 1930
Sinclair Lewis
The Nobel Prize in Literature 1930
Sinclair Lewis
Prize share: 1/1
The Nobel Prize in Literature 1930 was awarded t...


In [5]:
print(f"\n🔨 Building Knowledge Base from TriviaQA evidence passages...")
print(f"   Using {KB_SAMPLES:,} training samples → extracting their search_context passages")

docs    = []
skipped = 0

for sample in tqdm(triviaqa['train'].select(range(KB_SAMPLES)), desc="Building KB"):
    passages = sample['search_results']['search_context']
    question = sample['question']
    answer   = sample['answer']['value']

    for passage in passages:
        if not passage or len(passage.strip()) < 50:
            skipped += 1
            continue
        content = (
            f"Question: {question}\n"
            f"Answer: {answer}\n"
            f"Context: {passage.strip()[:800]}"
        )
        docs.append(Document(
            page_content=content,
            metadata={"source": "TriviaQA", "question": question[:80]}
        ))

print(f"\n   ✅ {len(docs):,} documents built from {KB_SAMPLES:,} samples")
print(f"   ⏭️  {skipped:,} passages skipped (too short)")
print(f"\n📄 Sample document:")
print(docs[0].page_content[:400])


🔨 Building Knowledge Base from TriviaQA evidence passages...
   Using 100,000 training samples → extracting their search_context passages


Building KB: 100%|██████████| 100000/100000 [01:05<00:00, 1528.35it/s]



   ✅ 412,973 documents built from 100,000 samples
   ⏭️  5 passages skipped (too short)

📄 Sample document:
Question: Which American-born Sinclair won the Nobel Prize for Literature in 1930?
Answer: Sinclair Lewis
Context: The Nobel Prize in Literature 1930
The Nobel Prize in Literature 1930
Sinclair Lewis
The Nobel Prize in Literature 1930
Sinclair Lewis
Prize share: 1/1
The Nobel Prize in Literature 1930 was awarded to Sinclair Lewis "for his vigorous and graphic art of description and his ability to 


## Build FAISS Vector Store

In [6]:
import shutil

print(f"📥 Loading embedding model: {EMBED_MODEL}")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": device},
    encode_kwargs={
        "normalize_embeddings": True,   # required for BGE models
        "batch_size": 128,
    },
)
print("✅ Embedding model ready (dim=768 for BGE-base)")

if os.path.exists(FAISS_PATH):
    print(f"\n📂 Loading existing FAISS index from {os.path.abspath(FAISS_PATH)}...")
    vectorstore = FAISS.load_local(
        FAISS_PATH, embeddings, allow_dangerous_deserialization=True,
    )
    print("✅ FAISS index loaded!")
else:
    print(f"\n🔨 Building FAISS index from {len(docs):,} documents...")
    t0 = datetime.now()
    vectorstore = FAISS.from_documents(docs, embeddings)
    print(f"   Built in {datetime.now()-t0}")
    vectorstore.save_local(FAISS_PATH)
    print(f"   💾 Saved → {os.path.abspath(FAISS_PATH)}")

# ── Reranker ───────────────────────────────────────────────────────────────────
print(f"\n📥 Loading reranker: {RERANKER_MODEL}")
reranker = CrossEncoder(RERANKER_MODEL, device=device, max_length=512)
print("✅ Reranker ready")

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K_RERANK})

# ── Retrieval test ─────────────────────────────────────────────────────────────
test_q = "Who wrote Romeo and Juliet?"
candidates = vectorstore.similarity_search(test_q, k=TOP_K_RERANK)
pairs      = [[test_q, doc.page_content] for doc in candidates]
scores     = reranker.predict(pairs)
reranked   = [doc for _, doc in sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)][:TOP_K]

print(f"\n🔍 Retrieval test: '{test_q}'")
print(f"   Before rerank — top doc: {candidates[0].page_content[:150]}...")
print(f"   After  rerank — top doc: {reranked[0].page_content[:150]}...")
print(f"\n✅ Retriever + Reranker ready — fetching top {TOP_K_RERANK}, reranking to {TOP_K}")

C:\Users\akhil\AppData\Local\Temp\ipykernel_6728\2055372502.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


📥 Loading embedding model: BAAI/bge-base-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\akhil\anaconda3\envs\squad-rag\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\akhil\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Reranker ready

🔍 Retrieval test: 'Who wrote Romeo and Juliet?'
   Before rerank — top doc: Question: A tragedy written early in the career of William Shakespeare about two young star-crossed lovers whose deaths ultimately reconcile their feu...
   After  rerank — top doc: Question: A tragedy written early in the career of William Shakespeare about two young star-crossed lovers whose deaths ultimately reconcile their feu...

✅ Retriever + Reranker ready — fetching top 10, reranking to 5


## Load Fine-Tuned Model (from baseline.ipynb)

In [7]:
from transformers import BitsAndBytesConfig
import gc

gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free before loading: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

print("📥 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

if os.path.exists(MERGED_DIR):
    print(f"📥 Loading pre-merged fine-tuned model from {os.path.abspath(MERGED_DIR)}...")

    model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR,
        quantization_config=bnb_config,
        device_map="cuda:0",
        trust_remote_code=True,
        attn_implementation="eager",
    )
    model_source = "MERGED_DIR (fine-tuned on SQuAD v2)"
    print("   ✅ Pre-merged model loaded")

elif os.path.exists(ADAPTER_DIR):
    print(f"📥 Loading base model + LoRA adapters from {os.path.abspath(ADAPTER_DIR)}...")

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,   # load in float16 first so merge works
        device_map="cuda:0",
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        attn_implementation="eager",
    )
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    model = model.merge_and_unload()  # merge before quantizing
    
    # re-quantize after merge
    model = model.to(torch.float16)
    model_source = "ADAPTER_DIR (fine-tuned on SQuAD v2, merged on-the-fly)"
    print("   ✅ Adapters merged into base model")

else:
    raise FileNotFoundError(
        f"❌ No fine-tuned model found!\n"
        f"   Expected MERGED_DIR  : {os.path.abspath(MERGED_DIR)}\n"
        f"   Expected ADAPTER_DIR : {os.path.abspath(ADAPTER_DIR)}\n"
        f"   Please run baseline.ipynb first!"
    )

model.config.use_cache = True
model.eval()

print(f"\n✅ Model ready!")
print(f"   Source    : {model_source}")
print(f"   VRAM used : {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"   VRAM free : {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print(f"   dtype     : {next(model.parameters()).dtype}")

VRAM free before loading: 6.54 GB
📥 Loading tokenizer...
📥 Loading pre-merged fine-tuned model from d:\content\qwen25-squadv2-merged...


Sliding Window Attention is enabled but not implemented for `eager`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

   ✅ Pre-merged model loaded

✅ Model ready!
   Source    : MERGED_DIR (fine-tuned on SQuAD v2)
   VRAM used : 2.65 GB
   VRAM free : 4.38 GB
   dtype     : torch.float16


## RAG Prompt & Inference Pipeline

In [8]:
from transformers import pipeline as hf_pipeline

gen_pipe = hf_pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
)
print("✅ Generation pipeline ready!")

RAG_SYSTEM = (
    "You are a helpful and precise question-answering assistant. "
    "You are given retrieved passages that may contain relevant information. "
    "Use the context to answer the question accurately and concisely. "
    "If the context does not support an answer, say so clearly."
)

def retrieve_context(question, k_fetch=TOP_K_RERANK, k_final=TOP_K):
    # Step 1: broad retrieval
    candidates = vectorstore.similarity_search(question, k=k_fetch)

    # Step 2: rerank
    pairs   = [[question, doc.page_content] for doc in candidates]
    scores  = reranker.predict(pairs)
    ranked  = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    top_docs = [doc for _, doc in ranked[:k_final]]

    # Step 3: format
    parts = []
    for i, doc in enumerate(top_docs, 1):
        parts.append(f"[Passage {i}]\n{doc.page_content}")
    return "\n\n".join(parts)

def generate_rag_answer(question, max_new_tokens=150):
    context = retrieve_context(question)

    prompt = (
        f"<|im_start|>system\n{RAG_SYSTEM}<|im_end|>\n"
        f"<|im_start|>user\n"
        f"--- RETRIEVED PASSAGES ---\n{context}\n--- END PASSAGES ---\n\n"
        f"Question: {question}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    out      = gen_pipe(prompt, max_new_tokens=max_new_tokens, do_sample=True,
                        temperature=0.1, top_p=0.9, repetition_penalty=1.1,
                        pad_token_id=tokenizer.eos_token_id)
    full     = out[0]["generated_text"]
    response = full.split("<|im_start|>assistant")[-1]
    response = response.replace("<|im_end|>", "").strip()
    return response, context

# Smoke test
print("\n🧪 Testing RAG pipeline with reranker...")
test     = triviaqa['validation'][0]
ans, ctx = generate_rag_answer(test['question'])
print(f"Q       : {test['question']}")
print(f"Context : {ctx[:200]}...")
print(f"Answer  : {ans[:200]}")
print(f"Correct : {test['answer']['value']}")
print("\n✅ RAG pipeline with reranker works!")

Device set to use cuda:0


✅ Generation pipeline ready!

🧪 Testing RAG pipeline with reranker...
Q       : Who was the man behind The Chipmunks?
Context : [Passage 1]
Question: Who are Alvin, Simon, Theodore, and Dave?
Answer: The Chipmunks
Context: Alvin Seville | Alvin and the Chipmunks Wiki | Fandom powered by Wikia
Work, failure, being alone, losing...
Answer  : Ross Bagdasarian, Sr.
Correct : David Seville

✅ RAG pipeline with reranker works!


## Full Inference on Evaluation Set

In [9]:
print(f"🔮 RAG inference — {EVAL_SAMPLES} samples (top-{TOP_K} TriviaQA passages each)...")

rag_results = []
val_subset  = triviaqa['validation'].select(range(EVAL_SAMPLES))

t0 = datetime.now()
for i, sample in enumerate(tqdm(val_subset, desc="RAG Inference")):
    question  = sample['question']
    reference = sample['answer']['value']
    generated, retrieved = generate_rag_answer(question)

    rag_results.append({
        "id":                i,
        "question":          question,
        "generated":         generated,
        "reference":         reference,
        "retrieved_context": retrieved,
    })

rag_df = pd.DataFrame(rag_results)
print(f"\n✅ Done in {datetime.now()-t0}!")
print(f"   Total samples : {len(rag_df)}")
print(f"Q        : {rag_df.iloc[0]['question']}")
print(f"Correct  : {rag_df.iloc[0]['reference']}")
print(f"Generated: {rag_df.iloc[0]['generated'][:200]}")

🔮 RAG inference — 500 samples (top-5 TriviaQA passages each)...


RAG Inference: 100%|██████████| 500/500 [13:15<00:00,  1.59s/it]


✅ Done in 0:13:15.838170!
   Total samples : 500
Q        : Who was the man behind The Chipmunks?
Correct  : David Seville
Generated: Ross Bagdasarian, Sr.


## ROUGE Evaluation

In [10]:
scorer     = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
r1, r2, rL = [], [], []

for _, row in tqdm(rag_df.iterrows(), total=len(rag_df), desc="ROUGE"):
    s = scorer.score(row['reference'], row['generated'].strip() or " ")
    r1.append(s['rouge1'].fmeasure)
    r2.append(s['rouge2'].fmeasure)
    rL.append(s['rougeL'].fmeasure)

rag_df['rouge1'] = r1
rag_df['rouge2'] = r2
rag_df['rougeL'] = rL

print(f"\n{'='*52}")
print(f"  📊 ROUGE — Model 2 (Qwen2.5-3B + RAG)")
print(f"{'='*52}")
print(f"  ROUGE-1 : {np.mean(r1):.4f}  ±{np.std(r1):.4f}")
print(f"  ROUGE-2 : {np.mean(r2):.4f}  ±{np.std(r2):.4f}")
print(f"  ROUGE-L : {np.mean(rL):.4f}  ±{np.std(rL):.4f}")
print(f"{'='*52}")

ROUGE: 100%|██████████| 500/500 [00:00<00:00, 8658.73it/s]


  📊 ROUGE — Model 2 (Qwen2.5-3B + RAG)
  ROUGE-1 : 0.5244  ±0.4685
  ROUGE-2 : 0.2454  ±0.4212
  ROUGE-L : 0.5230  ±0.4678


In [12]:
pip install bert-score

In [13]:
# ── Cell: BERTScore Evaluation ──────────────────────────────────────────────
# pip install bert-score (if not already installed)
# !pip install bert-score -q

from bert_score import score as bert_score_fn

print("🔮 Computing BERTScore (semantic similarity vs gold answers)...")
print("   Using model: microsoft/deberta-xlarge-mnli (best for QA)")

# Extract lists
candidates  = rag_df['generated'].apply(lambda x: x.strip() or " ").tolist()
references  = rag_df['reference'].tolist()

# Compute BERTScore — lang='en' auto-selects the best model
P, R, F1 = bert_score_fn(
    candidates,
    references,
    lang="en",
    model_type="microsoft/deberta-xlarge-mnli",
    batch_size=16,
    device=device,
    verbose=True,
)

rag_df['bertscore_P']  = P.numpy()
rag_df['bertscore_R']  = R.numpy()
rag_df['bertscore_F1'] = F1.numpy()

mean_P  = P.mean().item()
mean_R  = R.mean().item()
mean_F1 = F1.mean().item()

print(f"\n{'='*56}")
print(f"  📊 BERTScore — Model 2 (Qwen2.5-3B + RAG)")
print(f"{'='*56}")
print(f"  Precision : {mean_P:.4f}  ±{P.std().item():.4f}")
print(f"  Recall    : {mean_R:.4f}  ±{R.std().item():.4f}")
print(f"  F1        : {mean_F1:.4f}  ±{F1.std().item():.4f}")
print(f"{'='*56}")
print(f"  💡 BERTScore F1 > ROUGE because it captures semantic")
print(f"     similarity, not just exact n-gram overlap.")

🔮 Computing BERTScore (semantic similarity vs gold answers)...
   Using model: microsoft/deberta-xlarge-mnli (best for QA)


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

c:\Users\akhil\anaconda3\envs\squad-rag\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\akhil\.cache\huggingface\hub\models--microsoft--deberta-xlarge-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/51 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/32 [00:00<?, ?it/s]

done in 8.51 seconds, 58.79 sentences/sec

  📊 BERTScore — Model 2 (Qwen2.5-3B + RAG)
  Precision : 0.7738  ±0.2116
  Recall    : 0.7869  ±0.2001
  F1        : 0.7785  ±0.2059
  💡 BERTScore F1 > ROUGE because it captures semantic
     similarity, not just exact n-gram overlap.


## Faithfulness / Hallucination Evaluation

In [14]:
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

print("🧠 Loading NLI model...")
nli = CrossEncoder("cross-encoder/nli-deberta-v3-small", device=device, max_length=512)
print("✅ NLI model ready")

def faith_batch(premises, hypotheses, bs=32):
    scores = []
    for i in range(0, len(premises), bs):
        pairs  = list(zip(premises[i:i+bs], hypotheses[i:i+bs]))
        logits = nli.predict(pairs, apply_softmax=True)
        scores.extend(logits[:, 1].tolist())
    return scores

# Faithfulness vs gold reference answer
print("\n🔮 Computing reference faithfulness (generated vs gold answer)...")
ref_scores = faith_batch(rag_df['reference'].tolist(), rag_df['generated'].tolist())
rag_df['ref_faith']  = ref_scores
rag_df['ref_halluc'] = rag_df['ref_faith'] < 0.3

# Faithfulness vs retrieved context (grounding check)
print("🔮 Computing source faithfulness (generated vs retrieved passages)...")
trunc_ctx  = rag_df['retrieved_context'].apply(lambda x: x[:500]).tolist()
src_scores = faith_batch(trunc_ctx, rag_df['generated'].tolist())
rag_df['src_faith']  = src_scores
rag_df['src_halluc'] = rag_df['src_faith'] < 0.3

mean_ref   = np.mean(ref_scores);  halluc_ref = rag_df['ref_halluc'].mean()
mean_src   = np.mean(src_scores);  halluc_src = rag_df['src_halluc'].mean()

print(f"\n{'='*56}")
print(f"  🧠 FAITHFULNESS — Model 2 (Qwen2.5-3B + RAG)")
print(f"{'='*56}")
print(f"  Reference faithfulness (vs gold answer):")
print(f"    Mean score        : {mean_ref:.4f}")
print(f"    Faithful  (≥0.3)  : {(1-halluc_ref)*100:.1f}%")
print(f"    Hallucinated(<0.3): {halluc_ref*100:.1f}%")
print(f"  Source faithfulness (vs retrieved context):")
print(f"    Mean score        : {mean_src:.4f}")
print(f"    Grounded  (≥0.3)  : {(1-halluc_src)*100:.1f}%")
print(f"    Not grounded(<0.3): {halluc_src*100:.1f}%")
print(f"{'='*56}")

VRAM free: 4.48 GB
🧠 Loading NLI model...
✅ NLI model ready

🔮 Computing reference faithfulness (generated vs gold answer)...
🔮 Computing source faithfulness (generated vs retrieved passages)...

  🧠 FAITHFULNESS — Model 2 (Qwen2.5-3B + RAG)
  Reference faithfulness (vs gold answer):
    Mean score        : 0.5311
    Faithful  (≥0.3)  : 55.0%
    Hallucinated(<0.3): 45.0%
  Source faithfulness (vs retrieved context):
    Mean score        : 0.0483
    Grounded  (≥0.3)  : 4.6%
    Not grounded(<0.3): 95.4%


## Visualisation — Model 2 (RAG)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Model 2: Qwen2.5-3B + RAG (TriviaQA KB) — Full Evaluation',
             fontsize=15, fontweight='bold')

# ROUGE bars
ax = axes[0, 0]
means = [np.mean(r1), np.mean(r2), np.mean(rL)]
stds  = [np.std(r1),  np.std(r2),  np.std(rL)]
bars  = ax.bar(['ROUGE-1','ROUGE-2','ROUGE-L'], means, yerr=stds,
               color=['#2196F3','#4CAF50','#FF9800'], capsize=5, edgecolor='black', alpha=0.85)
ax.set_ylim(0, 1); ax.set_title('ROUGE Scores', fontweight='bold'); ax.set_ylabel('F1 Score')
ax.axhline(0.4, color='red', linestyle='--', alpha=0.5, label='Good threshold')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
for b, v in zip(bars, means):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.3f}', ha='center', fontweight='bold', fontsize=10)

# BERTScore bars
ax = axes[0, 1]
bs_means = [mean_P, mean_R, mean_F1]
bs_stds  = [P.std().item(), R.std().item(), F1.std().item()]
bs_bars  = ax.bar(['Precision','Recall','F1'], bs_means, yerr=bs_stds,
                  color=['#9C27B0','#E91E63','#3F51B5'], capsize=5, edgecolor='black', alpha=0.85)
ax.set_ylim(0, 1); ax.set_title('BERTScore (Semantic Similarity)', fontweight='bold')
ax.set_ylabel('Score'); ax.grid(axis='y', alpha=0.3)
for b, v in zip(bs_bars, bs_means):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.3f}', ha='center', fontweight='bold', fontsize=10)

# Faithful vs Hallucinated pie (vs gold reference)
ax = axes[0, 2]
ax.pie([1-halluc_ref, halluc_ref],
       labels=[f'Faithful\n{(1-halluc_ref)*100:.1f}%', f'Hallucinated\n{halluc_ref*100:.1f}%'],
       colors=['#4CAF50','#F44336'], autopct='%1.1f%%', startangle=90,
       wedgeprops={'edgecolor':'white','linewidth':2})
ax.set_title('Faithful vs Hallucinated (vs Gold)', fontweight='bold')

# Faithfulness histogram (vs gold reference)
ax = axes[1, 0]
ax.hist(ref_scores, bins=20, color='#9C27B0', edgecolor='black', alpha=0.8)
ax.axvline(0.3,     color='red',  linestyle='--', lw=2, label='Hallucination threshold')
ax.axvline(mean_ref, color='blue', linestyle='-',  lw=2, label=f'Mean = {mean_ref:.3f}')
ax.set_title('Reference Faithfulness Distribution', fontweight='bold')
ax.set_xlabel('Faithfulness Score'); ax.set_ylabel('Count')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

# Source grounding histogram
ax = axes[1, 1]
ax.hist(src_scores, bins=20, color='#00BCD4', edgecolor='black', alpha=0.8)
ax.axvline(0.3,     color='red',  linestyle='--', lw=2, label='Grounding threshold')
ax.axvline(mean_src, color='blue', linestyle='-',  lw=2, label=f'Mean = {mean_src:.3f}')
ax.set_title('Grounding vs Retrieved Context', fontweight='bold')
ax.set_xlabel('Source Faithfulness Score'); ax.set_ylabel('Count')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

# All metrics summary bar
ax = axes[1, 2]
mnames  = ['ROUGE-1', 'ROUGE-L', 'BERTScore F1', 'Faithful']
mvalues = [np.mean(r1)*100, np.mean(rL)*100, mean_F1*100, (1-halluc_ref)*100]
bars3 = ax.bar(mnames, mvalues, color=['#2196F3','#FF9800','#3F51B5','#4CAF50'], edgecolor='black', alpha=0.85)
ax.set_ylim(0, 100); ax.set_title('All Metrics (%)', fontweight='bold')
ax.set_ylabel('Score (%)'); ax.grid(axis='y', alpha=0.3)
for b, v in zip(bars3, mvalues):
    ax.text(b.get_x()+b.get_width()/2, v+0.8, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
os.makedirs(SAVE_DIR, exist_ok=True)
plt.savefig(f"{SAVE_DIR}/model2_results.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"📊 Saved → {os.path.abspath(SAVE_DIR)}/model2_results.png")


## Model 1 vs Model 2 — Final Comparison

In [16]:
try:
    with open(MODEL1_METRICS_PATH) as f:
        m1 = json.load(f)
    m1_r1, m1_r2, m1_rL = m1['rouge']['rouge1'], m1['rouge']['rouge2'], m1['rouge']['rougeL']
    m1_faith  = m1['faithfulness']['mean_score']
    m1_halluc = m1['faithfulness']['hallucination_rate']
    m1_bs_f1  = m1.get('bertscore', {}).get('f1', None)
    print(f"✅ Loaded Model 1 metrics from {os.path.abspath(MODEL1_METRICS_PATH)}")
except FileNotFoundError:
    print(f"⚠️  Not found: {os.path.abspath(MODEL1_METRICS_PATH)}")
    print("   Run baseline.ipynb first — using placeholders for display")
    m1_r1, m1_r2, m1_rL = 0.35, 0.18, 0.30
    m1_faith, m1_halluc  = 0.45, 0.45
    m1_bs_f1             = None

m2_r1, m2_r2, m2_rL = np.mean(r1), np.mean(r2), np.mean(rL)
m2_faith  = mean_ref
m2_halluc = halluc_ref
m2_bs_f1  = mean_F1

print(f"\n{'='*66}")
print(f"  📊 FINAL COMPARISON: Model 1 (No RAG) vs Model 2 (RAG)")
print(f"  Fine-tune: SQuAD v2  |  RAG KB: TriviaQA passages")
print(f"{'='*66}")
print(f"  {'Metric':<30} {'Model 1':>9} {'Model 2':>9} {'Δ (M2-M1)':>10}")
print(f"  {'─'*60}")

for label, v1, v2 in [
    ("ROUGE-1",            m1_r1,    m2_r1),
    ("ROUGE-2",            m1_r2,    m2_r2),
    ("ROUGE-L",            m1_rL,    m2_rL),
    ("Faithfulness Score", m1_faith, m2_faith),
]:
    print(f"  {label:<30} {v1:>9.4f} {v2:>9.4f} {v2-v1:>+10.4f}")

if m1_bs_f1 is not None:
    print(f"  {'BERTScore F1 (semantic)':<30} {m1_bs_f1:>9.4f} {m2_bs_f1:>9.4f} {m2_bs_f1-m1_bs_f1:>+10.4f}")
else:
    print(f"  {'BERTScore F1 (semantic)':<30} {'N/A':>9} {m2_bs_f1:>9.4f} {'(no M1 baseline)':>10}")

print(f"  {'─'*60}")
print(f"  {'Hallucination Rate':<30} {m1_halluc*100:>8.1f}% {m2_halluc*100:>8.1f}% {(m2_halluc-m1_halluc)*100:>+9.1f}%")
print(f"{'='*66}")
print(f"  Source Faithfulness (RAG grounding) : {mean_src:.4f}")
print(f"  Note: ↑ ROUGE/Faithfulness/BERTScore = better | ↓ Hallucination = better")

✅ Loaded Model 1 metrics from d:\RAG\model1_outputs\model1_metrics.json

  📊 FINAL COMPARISON: Model 1 (No RAG) vs Model 2 (RAG)
  Fine-tune: SQuAD v2  |  RAG KB: TriviaQA passages
  Metric                           Model 1   Model 2  Δ (M2-M1)
  ────────────────────────────────────────────────────────────
  ROUGE-1                           0.7629    0.5244    -0.2385
  ROUGE-2                           0.6053    0.2454    -0.3599
  ROUGE-L                           0.7619    0.5230    -0.2389
  Faithfulness Score                0.0506    0.5311    +0.4805
  BERTScore F1 (semantic)              N/A    0.7785 (no M1 baseline)
  ────────────────────────────────────────────────────────────
  Hallucination Rate                 95.0%     45.0%     -50.0%
  Source Faithfulness (RAG grounding) : 0.0483
  Note: ↑ ROUGE/Faithfulness/BERTScore = better | ↓ Hallucination = better


## Visualisation — Full Comparison

In [17]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Model 1 (No RAG) vs Model 2 (RAG) — Full Evaluation Comparison',
             fontsize=15, fontweight='bold')

# ROUGE comparison
ax = axes[0, 0]
x     = np.arange(3)
w     = 0.35
rouge_labels = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
m1_vals = [m1_r1, m1_r2, m1_rL]
m2_vals = [m2_r1, m2_r2, m2_rL]
b1 = ax.bar(x - w/2, m1_vals, w, label='Model 1 (No RAG)', color='#B0BEC5', edgecolor='black', alpha=0.85)
b2 = ax.bar(x + w/2, m2_vals, w, label='Model 2 (RAG)',    color='#2196F3', edgecolor='black', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(rouge_labels)
ax.set_ylim(0, 1); ax.set_title('ROUGE Scores: M1 vs M2', fontweight='bold')
ax.set_ylabel('F1 Score'); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
for b, v in zip(list(b1)+list(b2), m1_vals+m2_vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.3f}', ha='center', fontsize=8, fontweight='bold')

# Faithfulness comparison
ax = axes[0, 1]
cats  = ['Model 1\n(No RAG)', 'Model 2\n(RAG)']
faith = [m1_faith, m2_faith]
bars  = ax.bar(cats, faith, color=['#B0BEC5','#4CAF50'], edgecolor='black', alpha=0.85, width=0.4)
ax.set_ylim(0, 1); ax.set_title('Faithfulness Score', fontweight='bold')
ax.set_ylabel('Mean Faithfulness'); ax.axhline(0.3, color='red', linestyle='--', alpha=0.5, label='Threshold')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
for b, v in zip(bars, faith):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.4f}', ha='center', fontweight='bold', fontsize=12)

# Hallucination comparison
ax = axes[0, 2]
halluc_vals = [m1_halluc*100, m2_halluc*100]
bars2 = ax.bar(cats, halluc_vals, color=['#F44336','#FF9800'], edgecolor='black', alpha=0.85, width=0.4)
ax.set_ylim(0, 100); ax.set_title('Hallucination Rate (lower = better)', fontweight='bold')
ax.set_ylabel('Hallucination Rate (%)'); ax.grid(axis='y', alpha=0.3)
for b, v in zip(bars2, halluc_vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.8, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=12)

# Faithfulness distribution M2
ax = axes[1, 0]
ax.hist(ref_scores, bins=20, color='#9C27B0', edgecolor='black', alpha=0.8, label='Model 2 (RAG)')
ax.axvline(0.3,    color='red',  linestyle='--', lw=2, label='Hallucination threshold')
ax.axvline(mean_ref, color='blue', linestyle='-',  lw=2, label=f'Mean = {mean_ref:.3f}')
ax.set_title('Model 2 Faithfulness Distribution', fontweight='bold')
ax.set_xlabel('Faithfulness Score'); ax.set_ylabel('Count')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

# Source grounding (RAG-specific)
ax = axes[1, 1]
ax.hist(src_scores, bins=20, color='#00BCD4', edgecolor='black', alpha=0.8)
ax.axvline(0.3,    color='red',  linestyle='--', lw=2, label='Grounding threshold')
ax.axvline(mean_src, color='blue', linestyle='-',  lw=2, label=f'Mean = {mean_src:.3f}')
ax.set_title('Model 2: Grounding vs Retrieved Context', fontweight='bold')
ax.set_xlabel('Source Faithfulness Score'); ax.set_ylabel('Count')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

# Delta summary
ax = axes[1, 2]
delta_labels = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'Faithfulness']
deltas       = [m2_r1-m1_r1, m2_r2-m1_r2, m2_rL-m1_rL, m2_faith-m1_faith]
colors       = ['#4CAF50' if d >= 0 else '#F44336' for d in deltas]
bars3 = ax.bar(delta_labels, deltas, color=colors, edgecolor='black', alpha=0.85)
ax.axhline(0, color='black', lw=1)
ax.set_title('Δ RAG Improvement (M2 - M1)', fontweight='bold')
ax.set_ylabel('Delta Score'); ax.grid(axis='y', alpha=0.3)
for b, v in zip(bars3, deltas):
    ypos = v + 0.002 if v >= 0 else v - 0.008
    ax.text(b.get_x()+b.get_width()/2, ypos, f'{v:+.4f}', ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
os.makedirs(SAVE_DIR, exist_ok=True)
plt.savefig(f"{SAVE_DIR}/model2_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"📊 Saved → {os.path.abspath(SAVE_DIR)}/model2_comparison.png")

📊 Saved → d:\RAG\model2_outputs/model2_comparison.png


C:\Users\akhil\AppData\Local\Temp\ipykernel_6728\3555880924.py:78: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Save Results & Metrics

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
rag_df.to_csv(f"{SAVE_DIR}/model2_results.csv", index=False, escapechar="\\")

model2_metrics = {
    "model":            "Qwen2.5-3B-Instruct (Fine-tuned on SQuAD v2 + RAG TriviaQA KB)",
    "eval_samples":     EVAL_SAMPLES,
    "kb_samples":       KB_SAMPLES,
    "eval_dataset":     "TriviaQA validation",
    "kb_dataset":       "TriviaQA training passages",
    "finetune_dataset": "SQuAD v2",
    "rouge": {
        "rouge1":     round(float(np.mean(r1)), 4),
        "rouge2":     round(float(np.mean(r2)), 4),
        "rougeL":     round(float(np.mean(rL)), 4),
        "rouge1_std": round(float(np.std(r1)),  4),
        "rouge2_std": round(float(np.std(r2)),  4),
        "rougeL_std": round(float(np.std(rL)),  4),
    },
    "bertscore": {
        "precision":     round(mean_P,              4),
        "recall":        round(mean_R,              4),
        "f1":            round(mean_F1,             4),
        "precision_std": round(P.std().item(),      4),
        "recall_std":    round(R.std().item(),      4),
        "f1_std":        round(F1.std().item(),     4),
        "model":         "microsoft/deberta-xlarge-mnli",
    },
    "faithfulness": {
        "reference_mean":          round(float(mean_ref),    4),
        "reference_faithful_pct":  round((1-halluc_ref)*100, 2),
        "reference_halluc_pct":    round(halluc_ref*100,     2),
        "source_mean":             round(float(mean_src),    4),
        "source_grounded_pct":     round((1-halluc_src)*100, 2),
        "source_not_grounded_pct": round(halluc_src*100,     2),
        "samples_evaluated":       int(len(rag_df)),
    },
}

with open(f"{SAVE_DIR}/model2_metrics.json", 'w') as f:
    json.dump(model2_metrics, f, indent=2)

comparison = {
    "model1_no_rag": {
        "dataset":      "SQuAD v2 (eval)",
        "rouge1": m1_r1, "rouge2": m1_r2, "rougeL": m1_rL,
        "faithfulness": m1_faith, "hallucination": m1_halluc,
        "bertscore_f1": m1_bs_f1,
    },
    "model2_rag": {
        "finetune_dataset": "SQuAD v2",
        "kb_dataset":       "TriviaQA passages",
        "eval_dataset":     "TriviaQA validation",
        "rouge1":        round(m2_r1,    4),
        "rouge2":        round(m2_r2,    4),
        "rougeL":        round(m2_rL,    4),
        "faithfulness":  round(m2_faith, 4),
        "hallucination": round(m2_halluc,4),
        "bertscore_f1":  round(m2_bs_f1, 4),
    },
    "delta_model2_minus_model1": {
        "rouge1":        round(m2_r1    - m1_r1,    4),
        "rouge2":        round(m2_r2    - m1_r2,    4),
        "rougeL":        round(m2_rL    - m1_rL,    4),
        "faithfulness":  round(m2_faith - m1_faith, 4),
        "hallucination": round(m2_halluc- m1_halluc,4),
        "bertscore_f1":  round(m2_bs_f1 - m1_bs_f1, 4) if m1_bs_f1 is not None else None,
    },
    "note": "Positive delta = Model 2 better for ROUGE/faithfulness/BERTScore. Negative delta = Model 2 better for hallucination."
}

with open(f"{SAVE_DIR}/final_comparison.json", 'w') as f:
    json.dump(comparison, f, indent=2)

print(f"💾 Saved to: {os.path.abspath(SAVE_DIR)}")
print(f"   model2_results.csv")
print(f"   model2_metrics.json")
print(f"   final_comparison.json")
print()
print("=" * 52)
print("  ✅ MODEL 2 COMPLETE — SUMMARY")
print("=" * 52)
print(f"  ROUGE-1       : {model2_metrics['rouge']['rouge1']:.4f}")
print(f"  ROUGE-2       : {model2_metrics['rouge']['rouge2']:.4f}")
print(f"  ROUGE-L       : {model2_metrics['rouge']['rougeL']:.4f}")
print(f"  BERTScore F1  : {model2_metrics['bertscore']['f1']:.4f}")
print(f"  Faithful      : {model2_metrics['faithfulness']['reference_faithful_pct']:.1f}%")
print(f"  Hallucinate   : {model2_metrics['faithfulness']['reference_halluc_pct']:.1f}%")
print(f"  Grounded      : {model2_metrics['faithfulness']['source_grounded_pct']:.1f}%")
print("=" * 52)

💾 Saved to: d:\RAG\model2_outputs
   model2_results.csv
   model2_metrics.json
   final_comparison.json

  ✅ MODEL 2 COMPLETE — SUMMARY
  ROUGE-1       : 0.5244
  ROUGE-2       : 0.2454
  ROUGE-L       : 0.5230
  BERTScore F1  : 0.7785
  Faithful      : 55.0%
  Hallucinate   : 45.0%
  Grounded      : 4.6%


Error while downloading from https://huggingface.co/microsoft/deberta-xlarge-mnli/resolve/refs%2Fpr%2F10/model.safetensors: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...
Trying to resume download...
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /microsoft/deberta-xlarge-mnli/resolve/refs%2Fpr%2F10/model.safetensors (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 7f1481cd-08e8-465f-9402-efa8fcaff172)')' thrown while requesting GET https://huggingface.co/microsoft/deberta-xlarge-mnli/resolve/refs%2Fpr%2F10/model.safetensors
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /microsoft/deberta-xlarge-mnli/resolve/refs%2Fpr%2F10/model.safetensors (Caused by NameResol